# El costo de la distancia — Pipeline (una sola corrida)

**Indicador estrella:** para cada departamento y cultivo (maíz, soja), qué **% del valor bruto** se pierde en el flete a puerto.

Corre de punta a punta y en orden. Resuelve: mapa reproducible, corrección de Chascomús única y temprana, un solo corte de 311 departamentos en todas las etapas, orden OSRM→motor→mapa, y `assert` de consistencia entre etapas.

**Red:** la Fase 2 llama a OSRM público. Correr en Colab o entorno con salida a internet.

In [ ]:
import pandas as pd, numpy as np, geopandas as gpd, requests, time
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

CAMPANIA='2024/25'
OSRM_URL='http://router.project-osrm.org'
CHUNK, PAUSA = 80, 1.5
PATH_EST='estimaciones_limpio.csv'; PATH_ZIP='departamentos_ign.zip'; PATH_TARIFA='fetra_nacional_marzo_2026.csv'

## Fase 2 — Matriz de distancias
### 1. Producción + corrección de Chascomús

In [ ]:
est = pd.read_csv(PATH_EST, dtype={'codigo_indec':str})
est = est[est['campania']==CAMPANIA].copy()
# Chascomús: estimaciones trae 06217; el shapefile IGN usa 06218. Se corrige UNA vez, acá arriba.
est.loc[(est['provincia']=='BUENOS AIRES') & (est['departamento']=='CHASCOMUS'),'codigo_indec']='06218'
assert '06217' not in set(est['codigo_indec']), "Chascomús sin corregir"
n_deptos = est['codigo_indec'].nunique()
print(f"Campaña {CAMPANIA}: {len(est)} filas | {n_deptos} departamentos")

### 2. Geometría IGN → centroides (orígenes)

In [ ]:
gdf = gpd.read_file(f'zip://{PATH_ZIP}')
gdf['codigo_indec'] = gdf['in1'].astype(str).str.zfill(5)
deptos_prod = est[['codigo_indec','provincia','departamento']].drop_duplicates()
gdf_prod = gdf.merge(deptos_prod, on='codigo_indec', how='inner')
sin_geo = set(deptos_prod['codigo_indec']) - set(gdf_prod['codigo_indec'])
assert not sin_geo, f"Sin geometría: {sorted(sin_geo)}"
assert gdf_prod['codigo_indec'].nunique()==n_deptos
cen = gdf_prod.to_crs(5347).geometry.centroid.to_crs(4326)
origenes = gdf_prod[['codigo_indec','provincia','departamento']].copy()
origenes['lon']=cen.x.values; origenes['lat']=cen.y.values
print(f"Orígenes: {len(origenes)} departamentos")

### 3. Puertos de destino

Coordenadas sobre la **terminal de embarque** (salida al agua), no sobre el centroide del partido. Fuente: información portuaria oficial (argentina.gob.ar) y complejo PGSM.

In [ ]:
# (lon, lat) — formato OSRM. Punto = terminales de granos / salida al agua.
PUERTOS = {
    'Gran Rosario (San Lorenzo)': (-60.733, -32.717),   # Puerto Gral. San Martín, cordón up-river Paraná
    'Bahía Blanca (Ing. White)':  (-62.3074, -38.7834),  # muelles cerealeros Ing. White / Galván (oficial)
    'Quequén (Necochea)':         (-58.7135, -38.5713),  # margen Quequén, terminales de granos (oficial)
}
nombres_puertos = list(PUERTOS.keys())
coords_puertos  = list(PUERTOS.values())

### 4. Matriz OSRM (distancia por ruta)
> Requiere salida a `router.project-osrm.org`.

In [ ]:
resultados_osrm = []
n_chunks = int(np.ceil(len(origenes)/CHUNK))
for i in range(n_chunks):
    chunk = origenes.iloc[i*CHUNK:(i+1)*CHUNK]
    coords = list(zip(chunk['lon'], chunk['lat'])) + coords_puertos
    coords_str = ';'.join(f"{lo},{la}" for lo,la in coords)
    n_orig = len(chunk)
    destinos_idx = ';'.join(str(n_orig+j) for j in range(len(coords_puertos)))
    fuentes_idx  = ';'.join(str(j) for j in range(n_orig))
    url = f"{OSRM_URL}/table/v1/driving/{coords_str}?sources={fuentes_idx}&destinations={destinos_idx}&annotations=distance"
    data = requests.get(url, timeout=60).json()
    if data.get('code')!='Ok':
        print(f"chunk {i+1}/{n_chunks} FALLÓ: {data.get('message')}"); continue
    for k, fila in enumerate(data['distances']):
        row = {'codigo_indec': chunk.iloc[k]['codigo_indec']}
        for j, nom in enumerate(nombres_puertos):
            row[f'dist_km__{nom}'] = fila[j]/1000 if fila[j] is not None else np.nan
        resultados_osrm.append(row)
    print(f"chunk {i+1}/{n_chunks} ok ({n_orig})"); time.sleep(PAUSA)
matriz = pd.DataFrame(resultados_osrm)
print("matriz:", matriz.shape)

In [ ]:
cols_dist = [f'dist_km__{p}' for p in nombres_puertos]
matriz['dist_km'] = matriz[cols_dist].min(axis=1)
matriz['puerto_asignado'] = matriz[cols_dist].idxmin(axis=1).str.replace('dist_km__','',regex=False)
dist_final = origenes.merge(matriz, on='codigo_indec', how='left')
assert not dist_final['dist_km'].isna().any(), f"Sin distancia: {dist_final[dist_final.dist_km.isna()].codigo_indec.tolist()}"
assert dist_final['codigo_indec'].nunique()==n_deptos
dist_final.to_csv('distancias.csv', index=False)
print(f"distancias.csv: {dist_final['codigo_indec'].nunique()} departamentos")
print(dist_final['puerto_asignado'].value_counts().to_string())

## Fase 3 — Motor de costos

**Decisiones (cerradas):** tarifa FeTrA Nacional marzo 2026; TC BCRA A3500 31/03/2026 = $1.382,7578; pizarra maíz por puerto; pizarra soja solo Rosario aplicada a los 3 puertos (**caveat:** el mapa de soja es *puro flete*).

In [ ]:
TC=1382.7578
PIZARRA={'Maíz':{'Gran Rosario (San Lorenzo)':243652/TC,'Bahía Blanca (Ing. White)':198.55,'Quequén (Necochea)':195.28},
         'Soja':{p:465711/TC for p in nombres_puertos}}
tarifa = pd.read_csv(PATH_TARIFA)
km_arr, tar_arr = tarifa['km'].values, tarifa['tarifa_pesos_tn'].values
KM_MAX = km_arr.max()
flete_pesos_tn = lambda d: np.interp(d, km_arr, tar_arr)
print(("AVISO: excede tabla" if dist_final['dist_km'].max()>KM_MAX
       else f"OK: dist máx {dist_final['dist_km'].max():.1f} km ≤ tabla {KM_MAX} km"))

In [ ]:
assert not (set(est['codigo_indec']) - set(dist_final['codigo_indec'])), "Productores sin distancia"
filas=[]
for cult in ['Maíz','Soja']:
    prod_c = est[est['cultivo']==cult][['codigo_indec','produccion_tn']]
    df = dist_final.merge(prod_c, on='codigo_indec', how='inner')
    df['flete_usd_tn']=df['dist_km'].apply(flete_pesos_tn)/TC
    df['pizarra_usd_tn']=df['puerto_asignado'].map(PIZARRA[cult])
    df['pct_flete_sobre_bruto']=df['flete_usd_tn']/df['pizarra_usd_tn']*100
    df['valor_neto_usd_tn']=df['pizarra_usd_tn']-df['flete_usd_tn']
    df['flete_total_usd']=df['flete_usd_tn']*df['produccion_tn']
    df['valor_neto_total_usd']=df['valor_neto_usd_tn']*df['produccion_tn']
    df['cultivo']=cult; filas.append(df)
resultados = pd.concat(filas, ignore_index=True)
assert resultados[['flete_usd_tn','pizarra_usd_tn','pct_flete_sobre_bruto']].isnull().sum().sum()==0
resultados.to_csv('resultados.csv', index=False)
print(f"resultados.csv: {len(resultados)} filas | {resultados['cultivo'].value_counts().to_dict()}")

### Métricas ejecutivas (para el informe)

In [ ]:
# Promedio SIMPLE (cada depto pesa igual) vs PONDERADO por producción + flete total en USD
resumen=[]
for c in ['Maíz','Soja']:
    d=resultados[resultados.cultivo==c]
    resumen.append({'cultivo':c,
        'pct_simple':d.pct_flete_sobre_bruto.mean(),
        'pct_ponderado':np.average(d.pct_flete_sobre_bruto,weights=d.produccion_tn),
        'produccion_Mtn':d.produccion_tn.sum()/1e6,
        'flete_total_MUSD':d.flete_total_usd.sum()/1e6})
resumen=pd.DataFrame(resumen)
print(resumen.round(1).to_string(index=False))
print(f"\nFlete total a puerto (maíz+soja): USD {resultados['flete_total_usd'].sum()/1e6:,.0f} M")

# Ranking de valor perdido en flete por provincia
def wavg(s): return np.average(s, weights=resultados.loc[s.index,'produccion_tn'])
prov=(resultados.groupby('provincia')
      .agg(flete_MUSD=('flete_total_usd', lambda s:s.sum()/1e6),
           pct_flete_pond=('pct_flete_sobre_bruto', wavg))
      .sort_values('flete_MUSD',ascending=False))
print("\nTop 12 provincias por valor perdido en flete:")
print(prov.round(1).head(12).to_string())

### QC — casos extremos

In [ ]:
print(f"Rango %flete: {resultados.pct_flete_sobre_bruto.min():.1f}% a {resultados.pct_flete_sobre_bruto.max():.1f}%")
print("\nTop 5 más golpeados:")
print(resultados.nlargest(5,'pct_flete_sobre_bruto')[['departamento','provincia','cultivo','dist_km','pct_flete_sobre_bruto']].to_string(index=False))
print("\nTop 5 más protegidos:")
print(resultados.nsmallest(5,'pct_flete_sobre_bruto')[['departamento','provincia','cultivo','dist_km','pct_flete_sobre_bruto']].to_string(index=False))

## Fase 4 — Mapas

Escala común 0–55% para comparar maíz y soja. Gris = sin producción registrada del cultivo.

**Caveat metodológico:** el origen del flete es el **centroide geométrico** del partido. En departamentos extensos (NOA) el centroide puede caer lejos del punto real de acopio, sesgando la distancia; leer los valores del norte como orden de magnitud, no como precisión de km.

In [ ]:
semaforo = LinearSegmentedColormap.from_list('semaforo', ['#1a9850','#fee08b','#d73027'])
mapa_maiz = gdf_prod.merge(resultados[resultados.cultivo=='Maíz'][['codigo_indec','pct_flete_sobre_bruto']], on='codigo_indec')
mapa_soja = gdf_prod.merge(resultados[resultados.cultivo=='Soja'][['codigo_indec','pct_flete_sobre_bruto']], on='codigo_indec')
base_gris = gdf[gdf['codigo_indec'].str[:2].isin(gdf_prod['codigo_indec'].str[:2].unique())]
VMIN,VMAX = 0,55

fig, axes = plt.subplots(1,2,figsize=(20,12))
for gp,nombre,ax in [(mapa_maiz,'Maíz',axes[0]),(mapa_soja,'Soja',axes[1])]:
    base_gris.plot(color='#e8e8e8',edgecolor='#aaa',linewidth=0.15,ax=ax)
    gp.plot(column='pct_flete_sobre_bruto',cmap=semaforo,vmin=VMIN,vmax=VMAX,linewidth=0.15,
            edgecolor='#555',ax=ax,legend=True,
            legend_kwds={'label':'% del valor bruto que se come el flete','shrink':0.6})
    # Puertos etiquetados
    for nom,(lo,la) in PUERTOS.items():
        ax.scatter(lo,la,marker='*',s=220,c='#1f3b70',edgecolor='white',zorder=5)
        ax.annotate(nom.split(' (')[0],(lo,la),xytext=(4,4),textcoords='offset points',
                    fontsize=8,fontweight='bold',color='#1f3b70')
    ax.set_title(f'{nombre} — promedio {gp.pct_flete_sobre_bruto.mean():.1f}%',fontsize=13); ax.axis('off')
plt.suptitle('El costo de la distancia — % del valor bruto perdido en flete a puerto (marzo 2026)\n'
             'Escala común 0–55%  ·  ★ puertos de embarque  ·  gris = sin producción del cultivo',fontsize=14,y=0.99)
plt.tight_layout(); plt.savefig('mapa_flete.png',dpi=200,bbox_inches='tight'); plt.show()
print("mapa_flete.png guardado")

### Mapa interactivo (HTML — para portfolio / GitHub Pages)

In [ ]:
import folium, branca
gp_int = gdf_prod.copy()
gp_int['geometry'] = gp_int['geometry'].simplify(0.01, preserve_topology=True)  # aligerar HTML
cmap = branca.colormap.LinearColormap(['#1a9850','#fee08b','#d73027'],vmin=0,vmax=55,
        caption='% del valor bruto que se come el flete a puerto')
m = folium.Map(location=[-33,-63], zoom_start=5, tiles='CartoDB positron')
for cult,show in [('Maíz',True),('Soja',False)]:
    fg = folium.FeatureGroup(name=cult, show=show)
    rc = resultados[resultados.cultivo==cult].set_index('codigo_indec')
    g = gp_int[gp_int['codigo_indec'].isin(rc.index)].copy()
    for col in ['pct_flete_sobre_bruto','dist_km','puerto_asignado','valor_neto_usd_tn']:
        g[col]=g['codigo_indec'].map(rc[col])
    g['nombre']=g['departamento']
    folium.GeoJson(g.to_json(),
        style_function=lambda f:{'fillColor':cmap(f['properties']['pct_flete_sobre_bruto']),
                                 'color':'#555','weight':0.3,'fillOpacity':0.85},
        tooltip=folium.GeoJsonTooltip(
            fields=['nombre','pct_flete_sobre_bruto','dist_km','puerto_asignado','valor_neto_usd_tn'],
            aliases=['Departamento','% flete/bruto','Distancia (km)','Puerto','Neto USD/tn'],localize=True)
    ).add_to(fg); fg.add_to(m)
pg=folium.FeatureGroup(name='Puertos',show=True)
for nom,(lo,la) in PUERTOS.items():
    folium.Marker([la,lo],tooltip=nom,icon=folium.Icon(color='blue',icon='ship',prefix='fa')).add_to(pg)
pg.add_to(m); m.add_child(cmap); folium.LayerControl(collapsed=False).add_to(m)
m.save('mapa_interactivo.html'); print("mapa_interactivo.html guardado")